### Загрузка данных


In [31]:
import numpy as np
import scipy.stats as ss
import matplotlib.pyplot as plt

data = np.loadtxt('ivaniv.dat')

before = data[:, 0]
after = data[:, 1]
n = len(before)

### Проверяем поддержку и рейтинг
- Поддержка - количество людей поставивших больше 50
- Рейтинг - среднее значение рейтингов которые поставили люди

In [32]:
support_before = np.sum(before >= 50)
support_after = np.sum(after >= 50)

rating_before = np.mean(before)
rating_after = np.mean(after)

print(f"До: поддержка: {support_before}, рейтинг: {rating_before} \nПосле: поддержка: {support_after}, рейтинг: {rating_after}")

До: поддержка: 48, рейтинг: 72.12 
После: поддержка: 49, рейтинг: 69.24


### Если смотреть по чистым значениям то поддержка возросла, а рейтинг снизился.

### Разделяем поддержку по людям которые сменили мнение или остались с прежним
- a - поддерживали до и после
- b - поддерживали, а потом перестали
- c - не поддерживали, а потом стали
- d - не поддерживали ни до ни после

Это критерий Макнемара, им нужно воспользоваться так как выборки зависимые потому-что это одни и те же люди

In [33]:
support_before = before >= 50
support_after = after >= 50

a = np.sum(np.logical_and(support_before, support_after))
b = np.sum(np.logical_and(support_before, np.logical_not(support_after)))
c = np.sum(np.logical_and(np.logical_not(support_before), support_after))
d = np.sum(np.logical_and(np.logical_not(support_before), np.logical_not(support_after)))

print(f"{a}, {b}\n{c}, {d}")

48, 0
1, 1


### Проверяем pvalue для критерия Макнемара через функцию binomtest(k, n, p)
Где k - сколько людей сменили мнение, n - число людей сменивших мнение, p - вероятность перехода

In [34]:
p_value = ss.binomtest(int(b), int(b + c), 0.5).pvalue

if p_value < 0.05:
    print(f"p_value = {p_value}, Поддержка изменилась")
else:
    print(f"p_value = {p_value}, Поддержка не изменилась")

p_value = 1.0, Поддержка не изменилась


### Вопрос 2 изменился ли средний рейтинг
Используем T критерий потому-что СКО неизвестно

In [35]:
x = after - before
x_mean = x.mean()
m0 = 0
s0 = x.std()

T = (x_mean - m0) / (s0 / np.sqrt(n))
student = ss.t(n - 1)
p_value = (1 - student.cdf(abs(T))) * 2

print("T:", T, "pv:", p_value)

if p_value < 0.05:
    print("Рейтинг вырос" if x_mean > 0 else "Рейтинг упал")
else:
    print("Рейтинг не изменился")

T: -3.761856121159456 pv: 0.00045087092966356046
Рейтинг упал
